In [ ]:
import os
import json
import random
import math
from typing import List, Dict, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
import logging

from transformers import AutoTokenizer, AutoModelForCausalLM


# -------------------------------
# Configuration (edit as needed)
# -------------------------------
TRAIN_PATH = "/workspace/unsupervised-truth-probes/data/train_truthfulqa.json"
TEST_PATH  = "/workspace/unsupervised-truth-probes/data/test_truthfulqa.json"
MODEL_NAME = "gpt2"  # keep small for simplicity
SEED       = 42
LR         = 5e-5
EPOCHS     = 1
BATCH_SIZE = 4
FEW_SHOT_K = 4
CONSISTENCY_THRESHOLD = 0.0
MAXLEN     = 1024


# -------------------------------
# Reproducibility and device
# -------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# -------------------------------
# Model and tokenizer
# -------------------------------
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)

# token ids for " True" and " False" (leading space matters for GPT-2-like tokenizers)
true_id  = tokenizer(" True",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" False", add_special_tokens=False).input_ids[0]


# -------------------------------
# Data loading and normalization
# -------------------------------
print("Loading data...")
with open(TRAIN_PATH) as f:
    train_raw = json.load(f)
with open(TEST_PATH) as f:
    test_raw = json.load(f)

# Normalize to a consistent structure used throughout
# We keep optional fields if present, with safe fallbacks

def normalize_examples(raw: List[dict]) -> List[dict]:
    examples = []
    for ex in raw:
        question = ex.get("question", "")
        claim    = ex.get("choice", ex.get("cot", ""))
        label    = 1 if ex.get("label") else 0
        consistency_id = int(ex.get("consistency_id", -1))
        # group by textual answer if present, else fallback to ref_answer or claim
        answer_text = str(ex.get("answer", ex.get("ref_answer", claim)))
        examples.append({
            "question": question,
            "choice": claim,
            "label": label,
            "consistency_id": consistency_id,
            "answer_text": answer_text,
        })
    return examples

train_examples = normalize_examples(train_raw)
test_examples  = normalize_examples(test_raw)


# -------------------------------
# Prompt formatting utilities
# -------------------------------

def format_target(example: dict) -> str:
    return (
        f"Question: {example['question']}\n"
        f"Claim: {example['choice']}\n"
        "I think this claim is"
    )


def format_demo_with_label(example: dict, label_int: int) -> str:
    ans = " True" if int(label_int) == 1 else " False"
    return (
        f"Question: {example['question']}\n"
        f"Claim: {example['choice']}\n"
        f"I think this claim is{ans}\n"
    )


def build_random_k_shot_prefix(pool: List[dict], k: int) -> str:
    if k <= 0 or len(pool) == 0:
        return ""
    k = min(k, len(pool))
    sampled_indices = random.sample(range(len(pool)), k)
    demos: List[str] = []
    for idx in sampled_indices:
        rnd_label = random.randint(0, 1)
        demos.append(format_demo_with_label(pool[idx], rnd_label))
    return "\n".join(demos).rstrip("\n")


# -------------------------------
# Scoring: margins for True vs False
# -------------------------------
@torch.no_grad()
def score_margins(prompts: List[str], batch_size: int = 32) -> List[float]:
    margins: List[float] = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Scoring"):
        batch = prompts[i:i+batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAXLEN,
        ).to(device)
        logits = model(**enc).logits  # [B, T, V]
        lengths = enc["attention_mask"].sum(dim=1)  # [B]
        last_logits = logits[torch.arange(len(batch), device=logits.device), lengths - 1]  # [B, V]
        logp = torch.log_softmax(last_logits, dim=-1)  # [B, V]
        margins.extend((logp[:, true_id] - logp[:, false_id]).detach().cpu().tolist())
    return margins


# -------------------------------
# Consistency labeling
# -------------------------------
# We assign label 1 to the answer-group with the highest score for each consistency_id
# if it exceeds a threshold; others become 0. Examples with no consistency_id (-1)
# are treated as their own singleton group.

def build_group_ids(examples: List[dict]) -> Tuple[List[int], List[int]]:
    consistency_ids: List[int] = []
    group_ids: List[int] = []
    groups_by_cid: Dict[int, Dict[str, int]] = {}

    for ex in examples:
        cid = int(ex.get("consistency_id", -1))
        ans = str(ex.get("answer_text", ""))
        group_map = groups_by_cid.setdefault(cid, {})
        if ans not in group_map:
            group_map[ans] = len(group_map)
        consistency_ids.append(cid)
        group_ids.append(group_map[ans])
    return consistency_ids, group_ids


# Provided implementation (verbatim)
logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

def apply_consistency(
    scores: torch.Tensor,
    consistency_ids: list[int],
    consistency_groups: list[int],
    threshold: float = 0.0
) -> torch.Tensor:
    """
    Apply consistency across groups for each unique consistency ID.

    For each unique consistency ID, the group with the highest score above
    the given threshold is marked as the true group. All entries belonging
    to that group are assigned `1`, and others `0`. If the maximum score
    does not exceed the threshold, all entries are assigned `0`.

    Args:
        scores (torch.Tensor): Tensor of scores for each entry.
        consistency_ids (list[int]): List mapping each entry to a consistency ID.
        consistency_groups (list[int]): List mapping each entry to a group.
        threshold (float, optional): Minimum score required for assigning a
            true group. Defaults to 0.0.

    Returns:
        torch.Tensor: Long tensor of 0/1 values indicating consistency labels.
    """
    logger.info("Applying consistency to scores.")
    assert len(scores) == len(consistency_ids) == len(consistency_groups), (
        "scores, consistency_ids, and consistency_groups must have the same length"
    )
    results = []
    unique_ids = set(consistency_ids)
    for uid in unique_ids:
        subset_indices = [i for i, cid in enumerate(consistency_ids) if cid == uid]
        score_subset = scores[subset_indices]
        group_subset = [consistency_groups[i] for i in subset_indices]
        max_score, max_index = score_subset.max(0)
        if max_score > threshold:
            true_group = group_subset[max_index.item()]
            results.extend([group == true_group for group in group_subset])
        else:
            results.extend([False] * len(score_subset))
    logger.info("Consistency applied to scores.")
    return torch.tensor(results, dtype=torch.long, device=scores.device)


# -------------------------------
# Build prompts and pseudo-labels (random few-shot consistency)
# -------------------------------
base_train_prompts = [format_target(ex) for ex in train_examples]
kshot_prefix = build_random_k_shot_prefix(train_examples, FEW_SHOT_K)

if kshot_prefix:
    train_prompts_for_labeling = [kshot_prefix + "\n" + p for p in base_train_prompts]
else:
    train_prompts_for_labeling = base_train_prompts

print("Scoring train margins (with k-shot prefix for labeling)...")
train_margins = score_margins(train_prompts_for_labeling, batch_size=32)
train_margins_tensor = torch.tensor(train_margins, dtype=torch.float32, device=device)

consistency_ids, group_ids = build_group_ids(train_examples)
pseudo_labels = apply_consistency(
    scores=train_margins_tensor,
    consistency_ids=consistency_ids,
    consistency_groups=group_ids,
    threshold=CONSISTENCY_THRESHOLD,
)
print(f"Pseudo labels positive ratio: {pseudo_labels.float().mean().item():.4f}")


# -------------------------------
# Training loop (simple, single-epoch)
# -------------------------------
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# We'll train on base prompts (no k-shot prefix) to keep stimuli simple
train_loader = DataLoader(base_train_prompts, batch_size=BATCH_SIZE, shuffle=True)
prompt_to_index = {p: i for i, p in enumerate(base_train_prompts)}

running_loss = 0.0
num_steps = 0

print("Training...")
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    num_steps = 0
    for batch_prompts in tqdm(train_loader, desc=f"Epoch {epoch}"):
        idxs = [prompt_to_index[p] for p in batch_prompts]
        batch_labels = pseudo_labels[idxs]

        enc = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAXLEN,
        ).to(device)

        logits = model(**enc).logits  # [B, T, V]
        lengths = enc["attention_mask"].sum(dim=1)  # [B]
        last_logits = logits[torch.arange(logits.size(0), device=logits.device), lengths - 1]  # [B, V]

        # Build 2-class logits [False, True]
        pair_logits = torch.stack([last_logits[:, false_id], last_logits[:, true_id]], dim=1)  # [B, 2]
        loss = F.cross_entropy(pair_logits, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item())
        num_steps += 1

    avg_loss = running_loss / max(1, num_steps)
    print(f"Epoch {epoch} average loss: {avg_loss:.4f}")


# -------------------------------
# Evaluation on test set
# -------------------------------
model.eval()

base_test_prompts = [format_target(ex) for ex in test_examples]
print("Scoring test margins (zero-shot)...")
test_margins = score_margins(base_test_prompts, batch_size=32)

# Simple metrics: AUROC and accuracy at threshold 0
try:
    y_true_test = np.array([ex["label"] for ex in test_examples], dtype=int)
    y_score_test = np.array(test_margins, dtype=float)

    auroc = roc_auc_score(y_true_test, y_score_test)
    preds_at_zero = (y_score_test >= 0).astype(int)
    acc_at_zero = (preds_at_zero == y_true_test).mean()

    print(f"Test AUROC: {auroc:.4f}")
    print(f"Test Acc@0: {acc_at_zero:.4f}")

    # Optionally compute a max-accuracy threshold for reporting
    fpr, tpr, thr = roc_curve(y_true_test, y_score_test)
    accs = []
    for t in np.unique(np.concatenate([thr, np.array([-np.inf, np.inf])])):
        pr = (y_score_test >= t).astype(int)
        accs.append((pr == y_true_test).mean())
    print(f"Test MaxAcc: {np.max(accs):.4f}")
except Exception as e:
    print(f"[warn] Could not compute AUROC metrics: {e}")

print("Done.")
